# OTD 2025 + 2026 Upload

This notebook loads the configured sheets from the 2025 and 2026 workbooks, normalizes them into one OTD shape, adds a `Year` column, validates the result, and replaces the SQL OTD table.

Populate the two blank sheet-name lists in the first code cell before running the loader. Each sheet name is also used as that sheet's `BU` value. If one 2026 sheet stores Program in its second column, assign its name to `SECOND_COLUMN_PROGRAM_SHEET`.


In [ ]:
import os
import urllib

import pandas as pd
import pyodbc
from sqlalchemy import create_engine


FILE_2025 = "data/OTD_data_2025.xlsx"
FILE_2026 = "data/otd_data.xlsx"

# Fill these in on the computer where the source workbooks are available.
SHEET_NAMES_2025 = []
SHEET_NAMES_2026 = []
SECOND_COLUMN_PROGRAM_SHEET = ""


# Keep credentials out of the notebook itself.
server = os.environ["SQL_SERVER"]
database = os.environ["SQL_DATABASE"]
username = os.environ["SQL_USERNAME"]
password = os.environ["SQL_PASSWORD"]

conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "ENCRYPT=yes;"
    f"UID={username};"
    f"PWD={password};"
    "TrustServerCertificate=YES"
)

conn = pyodbc.connect(conn_str)
params = urllib.parse.quote_plus(conn_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


## Generic workbook loader

Every configured sheet follows the same rules: column zero becomes `Program` unless it is a 2026 sheet matching `SECOND_COLUMN_PROGRAM_SHEET`, in which case column one becomes `Program`. The timeline column is detected from its values, recognizable identifier and month headers are normalized, merged identifier cells are filled down, and missing optional columns are created. Any 2025 source row containing `Total` is discarded.

For each program/project group, a `Contract Commitment` row is used when at least one month is nonzero. If its monthly values are all zero or missing, `Factory Planned` supplies the commitment, with `Planned` retained as a fallback.


In [ ]:
MONTHS = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]

FINAL_COLUMNS = [
    "Timeline",
    "Program",
    "BU",
    "Project ID",
    "Site",
    "Type",
    *MONTHS,
    "Total",
    "Year",
]

TIMELINE_VALUES = {
    "contract commitment",
    "factory planned",
    "planned",
    "actual",
    "actual delivered",
    "actuals delivered",
}

TIMELINE_RENAMES = {
    "contract commitment": "Contract Commitment",
    "factory planned": "Contract Commitment",
    "planned": "Contract Commitment",
    "actual": "Actuals Delivered",
    "actual delivered": "Actuals Delivered",
    "actuals delivered": "Actuals Delivered",
}

MONTH_RENAMES = {
    "JAN": "Jan",
    "FEB": "Feb",
    "MAR": "Mar",
    "APR": "Apr",
    "APRIL": "Apr",
    "MAY": "May",
    "JUN": "Jun",
    "JUL": "Jul",
    "AUG": "Aug",
    "SEP": "Sep",
    "SEPT": "Sep",
    "OCT": "Oct",
    "NOV": "Nov",
    "DEC": "Dec",
}


def normalize_header(value):
    return str(value).strip().lower().replace("_", " ")


def detect_timeline_column(df, sheet_name):
    for column in df.columns:
        if normalize_header(column) == "timeline":
            return column

    best_column = None
    best_score = 0

    for column in df.columns:
        values = df[column].dropna().astype(str).str.strip().str.lower()
        score = values.isin(TIMELINE_VALUES).sum()

        if score > best_score:
            best_column = column
            best_score = score

    if best_column is None or best_score == 0:
        raise ValueError(
            f"Could not identify the timeline column on sheet {sheet_name!r}. "
            f"Columns found: {list(df.columns)}"
        )

    return best_column


def normalize_identifier_columns(df):
    rename_map = {}

    for column in df.columns:
        key = normalize_header(column)

        if key in {"project", "project id"}:
            rename_map[column] = "Project ID"
        elif key == "site":
            rename_map[column] = "Site"
        elif key == "type":
            rename_map[column] = "Type"

    return df.rename(columns=rename_map)


def normalize_month_columns(df):
    rename_map = {}

    for column in df.columns:
        month_name = MONTH_RENAMES.get(str(column).strip().upper())
        if month_name:
            rename_map[column] = month_name

    return df.rename(columns=rename_map)


def align_otd(df):
    df = df.copy()

    for column in ["Project ID", "Site", "Type"]:
        if column not in df.columns:
            df[column] = pd.NA

    for month in MONTHS:
        if month not in df.columns:
            df[month] = 0
        df[month] = pd.to_numeric(df[month], errors="coerce").fillna(0)

    df["Total"] = df[MONTHS].sum(axis=1)

    for column in FINAL_COLUMNS:
        if column not in df.columns:
            df[column] = pd.NA

    return df[FINAL_COLUMNS]


def select_commitment_rows(df):
    timeline = df["Timeline"].astype(str).str.strip().str.lower()
    identity_columns = [
        column for column in ["Program", "Project ID", "Site", "Type"]
        if column in df.columns
    ]
    group_keys = df[identity_columns].copy()

    for column in identity_columns:
        group_keys[column] = (
            group_keys[column].fillna("").astype(str).str.strip()
        )

    is_contract = timeline.eq("contract commitment")
    is_factory_planned = timeline.eq("factory planned")
    is_planned = timeline.eq("planned")
    contract_has_values = is_contract & df[MONTHS].ne(0).any(axis=1)

    selection = group_keys.copy()
    selection["_contract_has_values"] = contract_has_values.to_numpy()
    selection["_has_factory_planned"] = is_factory_planned.to_numpy()

    grouped = selection.groupby(identity_columns, dropna=False)
    group_has_contract_values = grouped["_contract_has_values"].transform("any")
    group_has_factory_planned = grouped["_has_factory_planned"].transform("any")

    is_commitment_candidate = is_contract | is_factory_planned | is_planned
    keep_commitment = (
        contract_has_values
        | (is_factory_planned & ~group_has_contract_values)
        | (is_planned & ~group_has_contract_values & ~group_has_factory_planned)
    )

    return df[~is_commitment_candidate | keep_commitment].copy()


def load_otd_sheet(path, sheet_name, year):
    df = pd.read_excel(path, sheet_name=sheet_name).dropna(how="all")

    comment_columns = [
        column for column in df.columns
        if normalize_header(column) == "comments"
    ]
    empty_helper_columns = [
        column for column in df.columns
        if "unnamed" in str(column).lower() and df[column].isna().all()
    ]
    df = df.drop(columns=comment_columns + empty_helper_columns, errors="ignore")

    if year == 2025:
        total_rows = df.astype("string").apply(
            lambda column: column.str.contains(
                "total", case=False, na=False, regex=False
            )
        ).any(axis=1)
        dropped_total_rows = int(total_rows.sum())
        df = df[~total_rows].copy()

        if dropped_total_rows:
            print(
                f"2025 / {sheet_name}: dropped "
                f"{dropped_total_rows:,} row(s) containing 'Total'"
            )

    if len(df.columns) == 0:
        raise ValueError(f"Sheet {sheet_name!r} has no usable columns.")

    program_column_index = (
        1 if year == 2026 and sheet_name == SECOND_COLUMN_PROGRAM_SHEET else 0
    )
    if len(df.columns) <= program_column_index:
        raise ValueError(
            f"Sheet {sheet_name!r} does not have column index "
            f"{program_column_index} for Program."
        )

    program_column = df.columns[program_column_index]
    other_program_columns = [
        column for index, column in enumerate(df.columns)
        if index != program_column_index and normalize_header(column) == "program"
    ]
    df = df.drop(columns=other_program_columns, errors="ignore")
    df.rename(columns={program_column: "Program"}, inplace=True)

    timeline_column = detect_timeline_column(df, sheet_name)
    if timeline_column != "Timeline":
        df.rename(columns={timeline_column: "Timeline"}, inplace=True)

    df = normalize_identifier_columns(df)
    df = normalize_month_columns(df)

    for column in ["Program", "Project ID", "Site", "Type"]:
        if column in df.columns:
            df[column] = df[column].ffill()

    for month in MONTHS:
        if month not in df.columns:
            df[month] = 0
        df[month] = pd.to_numeric(df[month], errors="coerce").fillna(0)

    df = select_commitment_rows(df)
    timeline = df["Timeline"].astype(str).str.strip().str.lower()
    df["Timeline"] = timeline.replace(TIMELINE_RENAMES)
    df = df[
        df["Timeline"].isin(["Contract Commitment", "Actuals Delivered"])
    ].copy()

    df["BU"] = sheet_name
    df["Year"] = year

    df = df[df["Program"].notna()]
    df = df[df["Program"].astype(str).str.strip().ne("")]

    return align_otd(df.reset_index(drop=True))


def load_otd_year(path, sheet_names, year):
    if not sheet_names:
        raise ValueError(f"Add the {year} sheet names to SHEET_NAMES_{year}.")

    frames = []

    for sheet_name in sheet_names:
        frame = load_otd_sheet(
            path,
            sheet_name=sheet_name,
            year=year,
        )
        print(f"{year} / {sheet_name}: {len(frame):,} normalized rows")
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)


def normalize_program_match_key(value):
    if pd.isna(value):
        return ""

    return "".join(
        character.casefold()
        for character in str(value)
        if character.isalnum()
    )


def enrich_2025_details_from_2026(otd_2025, otd_2026):
    detail_columns = ["Project ID", "Site", "Type", "BU"]
    lookup_source = otd_2026[
        otd_2026["Program"].notna()
    ][["Program", *detail_columns]].copy()
    lookup_source["_program_match_key"] = (
        lookup_source["Program"].map(normalize_program_match_key)
    )
    lookup_source = lookup_source[
        lookup_source["_program_match_key"].ne("")
    ].copy()

    for column in detail_columns:
        has_value = (
            lookup_source[column].notna()
            & lookup_source[column].astype(str).str.strip().ne("")
        )
        lookup_source[column] = lookup_source[column].where(has_value, pd.NA)

    value_counts = (
        lookup_source.groupby("_program_match_key", sort=False)[detail_columns]
        .nunique(dropna=True)
    )
    conflicting_keys = value_counts[(value_counts > 1).any(axis=1)].index.tolist()

    if conflicting_keys:
        conflicting_programs = (
            lookup_source[
                lookup_source["_program_match_key"].isin(conflicting_keys)
            ]["Program"]
            .drop_duplicates()
            .tolist()
        )
        preview = ", ".join(repr(value) for value in conflicting_programs[:10])
        suffix = "" if len(conflicting_programs) <= 10 else " ..."
        raise ValueError(
            "The following equivalent 2026 Program names have conflicting "
            f"Project ID, Site, Type, or BU details: {preview}{suffix}"
        )

    def first_nonblank(values):
        populated = values.dropna()
        return populated.iloc[0] if not populated.empty else pd.NA

    lookup = (
        lookup_source.groupby("_program_match_key", sort=False, as_index=False)
        .agg({column: first_nonblank for column in detail_columns})
    )
    enriched = otd_2025.copy()
    enriched["_program_match_key"] = (
        enriched["Program"].map(normalize_program_match_key)
    )
    enriched = enriched.merge(
        lookup,
        on="_program_match_key",
        how="left",
        suffixes=("", "_2026"),
        validate="many_to_one",
        indicator=True,
    )
    matched_program_count = enriched.loc[
        enriched["_merge"].eq("both"), "_program_match_key"
    ].nunique()

    for column in detail_columns:
        source_column = f"{column}_2026"
        enriched[column] = enriched[source_column].combine_first(enriched[column])
        enriched.drop(columns=source_column, inplace=True)

    enriched.drop(columns=["_merge", "_program_match_key"], inplace=True)
    print(
        f"Enriched 2025 details for {matched_program_count:,} "
        "alphanumeric Program match(es) from 2026"
    )

    return enriched[FINAL_COLUMNS]


## Load and combine both years


In [ ]:
otd_2025 = load_otd_year(
    FILE_2025,
    SHEET_NAMES_2025,
    2025,
)
otd_2026 = load_otd_year(
    FILE_2026,
    SHEET_NAMES_2026,
    2026,
)

otd_2025 = enrich_2025_details_from_2026(otd_2025, otd_2026)

otd = pd.concat([otd_2025, otd_2026], ignore_index=True)

print("2025 rows:", len(otd_2025))
print("2026 rows:", len(otd_2026))
print("Combined rows:", len(otd))
otd.head()


## Quick validation

Run these checks before replacing the SQL table.


In [ ]:
print("\nRows by year / BU / timeline:")
display(
    otd.groupby(["Year", "BU", "Timeline"])
       .size()
       .rename("Rows")
       .reset_index()
       .sort_values(["Year", "BU", "Timeline"])
)

print("\nDelivered / commitment totals by year:")
summary = (
    otd.groupby(["Year", "Timeline"])[MONTHS]
       .sum()
       .sum(axis=1)
       .rename("Units")
       .reset_index()
)
display(summary)

print("\nMissing identifying values:")
display(
    otd.groupby("Year")[["Program", "Project ID", "Site", "Type"]]
       .apply(lambda values: values.isna().sum())
)


## Upload

Run this cell only after validation. It replaces the existing `qmi.otd` table with the combined dataset.


In [ ]:
otd.to_sql(
    "otd",
    schema="",
    con=engine,
    if_exists="replace",
    index=False,
)

print(f"Uploaded {len(otd):,} rows to .otd")
